# Oracle AI Feature Demos

This notebook focuses on Oracle-specific persistence and memory features, using deterministic static Tavily-style results where useful.

Features shown:

- JSON payload and provenance metadata
- persistence caps and score thresholds
- URL upsert behavior
- semantic deduplication
- expired cache cleanup
- vector index helper

The final cleanup cell removes the demo rows created by this notebook.

## Setup

Run this cell first. It loads `.env`, connects to Oracle, prepares the demo table, and imports shared notebook helpers.

In [ ]:
from pathlib import Path
import sys

helper_path = None
for root in [Path.cwd(), *Path.cwd().parents]:
    candidate = root / "examples" / "oracle" / "demo_helpers.py"
    if candidate.exists():
        helper_path = candidate
        break

if helper_path is None:
    candidate = Path.cwd() / "demo_helpers.py"
    if candidate.exists():
        helper_path = candidate

if helper_path is None:
    raise RuntimeError("Could not find examples/oracle/demo_helpers.py. Start Jupyter from the repository root or examples/oracle.")

sys.path.insert(0, str(helper_path.parent))
from demo_helpers import *

print("Using helper:", helper_path)

## Static Tavily-style results

These static results make feature behavior repeatable without depending on live web result changes.

In [ ]:
class StaticTavilySearch:
    def __init__(self, results):
        self.results = [dict(result) for result in results]

    def search(self, query, max_results=5, **kwargs):
        return {"results": [dict(result) for result in self.results[:max_results]]}


STATIC_RESULTS = [
    {
        "title": "Oracle VECTOR memory architecture",
        "url": "https://example.com/oracle-vector-memory",
        "content": "Oracle stores Tavily results with VECTOR embeddings, JSON payloads, provenance metadata, and memory lifecycle fields for AI agents.",
        "score": 0.99,
    },
    {
        "title": "Tavily freshness cache pattern",
        "url": "https://example.com/tavily-freshness-cache",
        "content": "Tavily supplies fresh web results while Oracle keeps reusable cache and durable memory for later retrieval.",
        "score": 0.88,
    },
    {
        "title": "Lower score result",
        "url": "https://example.com/lower-score-result",
        "content": "A lower score result can be returned without being persisted when persist_score_threshold is configured.",
        "score": 0.30,
    },
]

## Choose feature demo queries

Edit these query strings if you want to run the feature demos with different labels. Cleanup uses the same values.

In [ ]:
FEATURE_QUERY = "Oracle Tavily AI feature persistence demo"
DEDUP_QUERY = "Oracle Tavily semantic deduplication feature demo"
CLEANUP_QUERY = "Oracle Tavily expired cache cleanup feature demo"

for query in [FEATURE_QUERY, DEDUP_QUERY, CLEANUP_QUERY]:
    delete_rows_for_query(query)

## 1. JSON, provenance, persistence cap, and score threshold

Only two rows should persist because `max_persisted_foreign=2` and `persist_score_threshold=0.8`.

In [ ]:
feature_client = make_client(
    "hybrid_search",
    persistence_depth="cache_plus_memory",
    oracle_upsert_key="source_url",
    max_persisted_foreign=2,
    persist_score_threshold=0.8,
    dedup_similarity_threshold=None,
)
feature_client.tavily = StaticTavilySearch(STATIC_RESULTS)

feature_results = feature_client.search(
    FEATURE_QUERY,
    max_results=3,
    max_local=0,
    max_foreign=3,
    save_foreign=True,
)

show_results("Static Tavily results returned to caller", feature_results)
show_persisted_rows(FEATURE_QUERY, "Rows persisted after caps and score threshold")

## 2. Inspect Oracle metadata

This SQL view proves that JSON/provenance/memory fields were written into Oracle.

In [ ]:
with connection.cursor() as cursor:
    cursor.execute(
        f"""
        SELECT SOURCE_URL,
               SOURCE_TITLE,
               RETRIEVAL_MODE,
               MEMORY_SCOPE,
               CACHE_HIT,
               QUERY_COUNT,
               JSON_VALUE(RAW_PAYLOAD, '$.provenance.retrieval_mode') AS RAW_MODE
        FROM {TABLE_NAME}
        WHERE RETRIEVAL_QUERY = :query
        ORDER BY SOURCE_URL
        """,
        query=FEATURE_QUERY,
    )
    metadata_rows = [
        {
            "source_url": row[0],
            "source_title": row[1],
            "retrieval_mode": row[2],
            "memory_scope": row[3],
            "cache_hit": row[4],
            "query_count": row[5],
            "raw_mode": row[6],
        }
        for row in cursor.fetchall()
    ]

display_table(
    metadata_rows,
    ["source_url", "source_title", "retrieval_mode", "memory_scope", "cache_hit", "query_count", "raw_mode"],
    "Oracle metadata written by save_foreign=True",
)

## 3. URL upsert prevents repeated inserts

Repeating the same static Tavily results should update existing `SOURCE_URL` rows instead of inserting duplicates.

In [ ]:
rows_before = count_rows_for_query(FEATURE_QUERY)
feature_client.search(
    FEATURE_QUERY,
    max_results=3,
    max_local=0,
    max_foreign=3,
    save_foreign=True,
)
rows_after = count_rows_for_query(FEATURE_QUERY)

print("Rows before repeated save:", rows_before)
print("Rows after repeated save:", rows_after)
print("Rows inserted on repeat:", rows_after - rows_before)
show_persisted_rows(FEATURE_QUERY, "Rows after source URL upsert")

## 4. Semantic deduplication skips near-duplicates

The same semantic result is saved twice. The second insert should be skipped when it is above the dedup threshold.

In [ ]:
dedup_client = make_client(
    "hybrid_search",
    persistence_depth="cache_plus_memory",
    dedup_similarity_threshold=0.95,
)
dedup_client.tavily = StaticTavilySearch([
    {
        "title": "Unique semantic deduplication row",
        "url": "https://example.com/unique-semantic-dedup-row",
        "content": "A unique Oracle semantic deduplication example stores one Tavily memory row and skips the repeated near-duplicate vector.",
        "score": 0.91,
    }
])

dedup_client.search(DEDUP_QUERY, max_local=0, max_foreign=1, save_foreign=True)
rows_after_first = count_rows_for_query(DEDUP_QUERY)
dedup_client.search(DEDUP_QUERY, max_local=0, max_foreign=1, save_foreign=True)
rows_after_second = count_rows_for_query(DEDUP_QUERY)

print("Rows after first insert:", rows_after_first)
print("Rows after repeated near-duplicate insert:", rows_after_second)
print("Rows inserted on repeat:", rows_after_second - rows_after_first)

## 5. Expired cache cleanup

`cleanup_cache()` deletes expired `cache_only` rows. It does not delete `cache_plus_memory` rows when memory metadata columns exist.

In [ ]:
cleanup_client = make_client(
    "freshness_cache",
    persistence_depth="cache_only",
    cache_ttl_seconds=1,
    cache_score_threshold=-1.0,
)
cleanup_client.tavily = StaticTavilySearch(STATIC_RESULTS[:1])
cleanup_client.search(CLEANUP_QUERY, max_local=0, max_foreign=1, save_foreign=True)
print("Rows before TTL expiry:", count_rows_for_query(CLEANUP_QUERY))

time.sleep(2)
deleted = cleanup_client.cleanup_cache()
print("Expired cache_only rows deleted:", deleted)
print("Rows after cleanup for this query:", count_rows_for_query(CLEANUP_QUERY))

## 6. Vector index helper

This shows the public helper you can call when you want Oracle to create or verify the configured vector index. The live creation call is opt-in because small local Oracle Free containers can run out of vector memory while creating indexes.

In [ ]:
if os.environ.get("ORACLE_RUN_VECTOR_INDEX_DEMO") == "1":
    created = feature_client.ensure_oracle_vector_index()
    print("Vector index created:", created)
else:
    print("Vector index helper available: feature_client.ensure_oracle_vector_index()")
    print("Skipped live index creation. Set ORACLE_RUN_VECTOR_INDEX_DEMO=1 to run it in an environment with enough Oracle vector memory.")

## Cleanup

Run this at the end. It deletes only the rows created by this feature notebook.

In [ ]:
for query in [FEATURE_QUERY, DEDUP_QUERY, CLEANUP_QUERY]:
    delete_rows_for_query(query)
print("Cleaned Oracle AI feature demo rows.")